In [ ]:
import numpy as np

def build_Q(costs, edges, nodes, targets, P1=20, P2=20, P3=20):
    source = [k for k, v in targets.items() if v == 1][0]
    sink   = [k for k, v in targets.items() if v == -1][0]

    intermediate_nodes = [v for v in nodes if v not in [source, sink]]

    num_helpers = len(intermediate_nodes)
    num_vars = len(costs) + num_helpers

    Q = np.zeros((num_vars, num_vars))
    

    # Add linear edge costs
    for i in range(len(costs)):
        Q[i, i] += costs[i]

    # Flow conservation constraints (P1)
    for v in nodes:
        conn = []

        for i, (u, w) in enumerate(edges):
            if u == v:
                conn.append((i, 1))      # outgoing
            elif w == v:
                conn.append((i, -1))     # incoming

        Tv = targets[v]

        for i, s_vi in conn:
            Q[i, i] += P1 * (1 - 2 * Tv * s_vi)

            for j, s_vj in conn:
                if i < j:
                    Q[i, j] += 2 * P1 * s_vi * s_vj

    # Degree constraints for intermediate nodes (P2)
    for offset, v in enumerate(intermediate_nodes):
        y_idx = len(costs) + offset

        conn = []
        for i, (u, w) in enumerate(edges):
            if u == v or w == v:
                conn.append(i)

        # edge-edge terms
        for i in conn:
            Q[i, i] += P2
            for j in conn:
                if i < j:
                    Q[i, j] += 2 * P2

        # edge-helper interaction
        for i in conn:
            Q[i, y_idx] -= 4 * P2

        # helper diagonal
        Q[y_idx, y_idx] += 4 * P2

    # Endpoint constraints (P3)

    # Source node A = node 0, exactly one outgoing edge
    edges_from_source = [i for i, (u, w) in enumerate(edges) if u == source]

    
    
    for i in edges_from_source:
        Q[i, i] += P3
        for j in edges_from_source:
            if i < j:
                Q[i, j] += 2 * P3
        Q[i, i] -= 2 * P3


    # Sink node D = node 3, exactly one incoming edge
    edges_to_sink = [i for i, (u, w) in enumerate(edges) if w == sink]

    for i in edges_to_sink:
        Q[i, i] += P3
        for j in edges_to_sink:
            if i < j:
                Q[i, j] += 2 * P3
        Q[i, i] -= 2 * P3

    return Q

: 

In [ ]:
edges = [
    (0,1),
    (0,2),
    (1,2),
    (1,3),
    (2,3),
    (2,4),
    (3,4)
]

edge_names = ["01","02","12","13","23","24","34"]

costs = [2,4,1,3,2,5,1]

nodes = [0,1,2,3,4]

targets = {
    0: 1,   # source
    4: -1,  # sink
    1: 0,
    2: 0,
    3: 0
}

Q = build_Q(costs, edges, nodes, targets)
print(Q.shape)
print(Q)

In [ ]:
from qat.opt import QUBO
# Build QUBO for SQA
Q = build_Q(costs, edges, nodes, targets)
# Make Q symmetric
Q = (Q + Q.T) / 2

print("Q matrix:")
print(Q)

In [ ]:
import numpy as np
from qat.qpus import SimulatedAnnealing
from qat.core import Variable
from collections import defaultdict

qubo = QUBO(Q)
# convert qubo to Ising model
ising = qubo.to_ising()
#create job for sqa
job = ising.sqa_job(nbshots=1000)

# Temperature schedule
t = Variable("t", float)
temp_t = 50* (1 - t) + 5

# Initialize SQA Solver
qpu = SimulatedAnnealing(temp_t=temp_t, n_steps=300)

result = qpu.submit(job)
print(result)

all_states = []
all_energies = []

# collect all samples
all_samples = []

for _ in range(10):   # run multiple times
    result = qpu.submit(job)

    for sample in result.raw_data:
        bitstring = ''.join(str(int(b)) for b in sample.state)
        bitstring = bitstring.zfill(Q.shape[0])

        x = np.array([int(b) for b in bitstring])
        energy = x @ Q @ x

        all_states.append(bitstring)
        all_energies.append(energy)

        all_samples.append(sample)

unique_states = set()

for sample in all_samples:

    # Convert state to bitstring
    bitstring = ''.join(str(int(b)) for b in sample.state)
    bitstring = bitstring.zfill(Q.shape[0])
    unique_states.add(bitstring)

print("Unique solutions:", len(unique_states))

# Extract solution
best_sample = None
best_energy = float('inf')


for sample in all_samples:
    bitstring = ''.join(str(int(b)) for b in sample.state)
    bitstring = bitstring.zfill(Q.shape[0])

    x = np.array([int(b) for b in bitstring])
    energy = x @ Q @ x

    if energy < best_energy:
        
        best_energy = energy
        best_sample = sample
        best_bitstring = bitstring

print("Best state:", best_bitstring)
print("Min energy:", best_energy)


# Decode edges
num_edges = len(edges)
edge_bits = best_bitstring[:num_edges]
selected_edges = [edges[i] for i, b in enumerate(edge_bits) if b == '1']
print("Selected edges:", selected_edges)

# Check valid path
def is_valid_path(selected_edges, source, sink):
    if not selected_edges:
        return False

    adj = defaultdict(list)
    for u, v in selected_edges:
        adj[u].append(v)

    current = source
    visited = set()

    while current in adj:
        if current in visited:
            return False
        visited.add(current)

        if len(adj[current]) != 1:
            return False

        current = adj[current][0]
        if current == sink:
            return True

    return False

source = [k for k, v in targets.items() if v == 1][0]
sink = [k for k, v in targets.items() if v == -1][0]

print("Valid path" if is_valid_path(selected_edges, source, sink) else "Invalid path")

In [ ]:
import matplotlib.pyplot as plt

energies = []
path_lengths = []
valid_flags = []

num_edges = len(edges)

for sample in all_samples:
    
    # Convert state → bitstring (safe for your version)
    bitstring = ''.join(str(int(b)) for b in sample.state)
    bitstring = bitstring.zfill(Q.shape[0])

    # Extract edges
    edge_bits = bitstring[:num_edges]
    selected_edges = [edges[i] for i, b in enumerate(edge_bits) if b == '1']

    # Path length
    path_length = sum(costs[i] for i, b in enumerate(edge_bits) if b == '1')

    # Validity
    valid = is_valid_path(selected_edges, source, sink)

    # Energy
    x = np.array([int(b) for b in bitstring])
    energy = x @ Q @ x

    energies.append(energy)
    path_lengths.append(path_length)
    valid_flags.append(valid)

# Valid solutions 
valid_x = [path_lengths[i] for i in range(len(energies)) if valid_flags[i]]
valid_y = [energies[i] for i in range(len(energies)) if valid_flags[i]]

# Invalid solutions 
invalid_x = [path_lengths[i] for i in range(len(energies)) if not valid_flags[i]]
invalid_y = [energies[i] for i in range(len(energies)) if not valid_flags[i]]

# Plot valid and invalid solutions
plt.scatter(valid_x, valid_y, color='green', label='Valid')
plt.scatter(invalid_x, invalid_y, color='red', label='Invalid')

plt.xlabel("Path Length")
plt.ylabel("Energy")
plt.title("Energy vs Path Length (SQA)")

plt.axhline(y=min(energies), linestyle='--', label='Min Energy')


plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

In [ ]:
# Create a QAOA job with 2 layers
job = ising.qaoa_job(2, False)

In [ ]:
from qat.qpus import get_default_qpu
#Initialize QPU
qpu = get_default_qpu()

result = qpu.submit(job)

print(result)
print("Optimal parameters:", result.parameter_map)

In [ ]:
bitstrings = []
probabilities = []

# get parameters
param_map = result.parameter_map

#  bind parameters to circuit
final_job = job.circuit.bind_variables(param_map).to_job(nbshots=1000)

final_result = qpu.submit(final_job)

for sample in final_result:
    bitstring = sample.state.bitstring.zfill(Q.shape[0])  

    bitstrings.append(bitstring)
    probabilities.append(sample.probability)

print(bitstrings)
print(probabilities)

In [ ]:
import matplotlib.pyplot as plt

costs_list = []
valid_flags = []

num_edges = len(edges)

for sample in final_result:

    bitstring = sample.state.bitstring.zfill(Q.shape[0])

    edge_bits = bitstring[:num_edges]
    selected_edges = [edges[i] for i, b in enumerate(edge_bits) if b == '1']

    path_cost = sum(costs[i] for i, b in enumerate(edge_bits) if b == '1')

    valid = is_valid_path(selected_edges, source, sink)

    costs_list.append(path_cost)
    valid_flags.append(valid)

# valid
valid_x = [c for c, v in zip(costs_list, valid_flags) if v]
valid_y = [p for p, v in zip(probabilities, valid_flags) if v]

# invalid
invalid_x = [c for c, v in zip(costs_list, valid_flags) if not v]
invalid_y = [p for p, v in zip(probabilities, valid_flags) if not v]

# Plot
plt.scatter(invalid_x, invalid_y, color='red', label='Invalid')
plt.scatter(valid_x, valid_y, color='green', label='Valid')

plt.xlabel("Path Cost")
plt.ylabel("Probability")
plt.title("QAOA: Cost vs Probability")

plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()